# Fase 2: Preprocesamiento e Ingeniería de Variables
## Sistema de Predicción de Factores de Riesgo en Adolescentes Salvadoreños (GSHS 2013)

**Propósito:** Este notebook toma el dataset limpio de la fase anterior y aplica técnicas de ingeniería de características. Se seleccionarán las variables predictoras (evitando la colinealidad y la fuga de datos), se imputarán los valores nulos, se codificarán las variables categóricas y se escalarán los datos para preparar las matrices finales de entrenamiento para los modelos de regresión y clasificación.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

print("Librerías de preprocesamiento cargadas exitosamente.")

Librerías de preprocesamiento cargadas exitosamente.


## 1. Carga de Datos y Separación de Variables (Q vs QN)

**¿Qué hace este bloque?**
Carga los datos procesados previamente. Luego, separa las variables predictoras ($X$) de las variables objetivo ($y$). Además, elimina las columnas redundantes y aquellas que causan *Data Leakage*.

**¿Por qué lo hacemos así?**
El dataset original contiene preguntas categóricas (`Q1` a `Q58`) y sus recodificaciones numéricas (`QN6` a `QN58`). Utilizar ambas simultáneamente generaría una **multicolinealidad perfecta**, arruinando los modelos (especialmente la regresión lineal o logística). Optaremos por conservar las preguntas originales (`Q`) y aplicarles codificación categórica más adelante, eliminando las `QN`.
Además, **es obligatorio** eliminar el Peso (`Q5`) y la Estatura (`Q4`) del conjunto de variables predictoras al predecir el IMC, así como la pregunta base del riesgo de salud mental (`Q25`).

In [2]:
# Carga del dataset limpio
df = pd.read_csv('../data/processed/SLV2013_Limpios_Targets.csv')

# Definición de las variables objetivo (Targets)
y_reg = df['IMC']
y_clf = df['Riesgo_Salud_Mental']

# Eliminamos: QN (recodificadas, redundantes con Q), variables auxiliares del diseño
# muestral, los predictores directos del IMC (Q4, Q5) y las variables que
# forman el target de clasificación (Q25 y QN24) para evitar data leakage.
columnas_a_eliminar = [col for col in df.columns if col.startswith('QN') or col.startswith('qn')]
columnas_a_eliminar.extend(['Q4', 'Q5', 'Q25', 'IMC', 'Riesgo_Salud_Mental'])
columnas_a_eliminar.extend(['weight', 'stratum', 'psu'])

# Matriz de características en crudo (sin transformaciones aún)
X = df.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"Dimensiones de X (Predictoras): {X.shape}")
print(f"Dimensiones de y_reg (Target Regresión): {y_reg.shape}")
print(f"Dimensiones de y_clf (Target Clasificación): {y_clf.shape}")
print(f"\nDistribución del target de clasificación:")
print(y_clf.value_counts().to_frame().assign(pct=lambda d: (d['count']/len(y_clf)*100).round(1)))

Dimensiones de X (Predictoras): (1915, 46)
Dimensiones de y_reg (Target Regresión): (1915,)
Dimensiones de y_clf (Target Clasificación): (1915,)

Distribución del target de clasificación:
                     count   pct
Riesgo_Salud_Mental             
0                     1605  83.8
1                      310  16.2


## 2. Imputación de Valores Nulos

**¿Qué hace este bloque?**
Rellena los valores faltantes (`NaN`) en la matriz de características utilizando la estrategia de la moda (el valor más frecuente).

**¿Por qué lo hacemos así?**
Dado que la inmensa mayoría de nuestras variables predictoras (`Q1`, `Q2`, etc.) representan respuestas a encuestas de opción múltiple (categorías ordinales o nominales representadas por números enteros), imputar con la media generaría valores decimales inexistentes en las opciones de respuesta. La moda preserva la estructura de distribución categórica de la encuesta poblacional.

In [3]:
# La imputación, codificación y escalado NO se aplican aquí sobre el dataset completo.
# Hacerlo antes del split filtra información de test hacia train (data leakage).
# Estas transformaciones se encapsularán en Pipelines de sklearn y se ajustarán
# exclusivamente sobre X_train durante el entrenamiento (notebook 03).
print("Transformaciones diferidas al Pipeline de sklearn. Sin operaciones fit aquí.")

Transformaciones diferidas al Pipeline de sklearn. Sin operaciones fit aquí.


## 3. Codificación de Variables Categóricas (One-Hot Encoding)

**¿Qué hace este bloque?**
Transforma las variables categóricas en múltiples columnas binarias (0 y 1).

**¿Por qué lo hacemos así?**
Aunque los datos de la encuesta son números (ej. 1, 2, 3, 4), no siempre tienen un orden matemático lógico (por ejemplo, 1=Manzana, 2=Naranja no implica que Naranja > Manzana). El `One-Hot Encoding` evita que los modelos asuman jerarquías falsas. Utilizamos el parámetro `drop_first=True` para evitar la trampa de las variables ficticias (colinealidad perfecta entre las nuevas columnas binarias generadas).

In [4]:
# One-Hot Encoding eliminado de esta etapa por la misma razón:
# pd.get_dummies sobre las 1915 filas ajusta las categorías con información
# de los conjuntos de validación y test. Queda delegado al Pipeline.
print("OHE delegado al Pipeline de sklearn. Sin operaciones fit aquí.")

OHE delegado al Pipeline de sklearn. Sin operaciones fit aquí.


## 4. Escalado de Datos

**¿Qué hace este bloque?**
Estandariza el rango de las características utilizando `RobustScaler`. También imputamos los Targets en caso de que existan registros donde el IMC no pudo calcularse.

**¿Por qué lo hacemos así?**
Algoritmos como la Regresión Logística y la Regresión Lineal son sensibles a la magnitud de las variables. Aunque con One-Hot Encoding la mayoría de los datos son 0 y 1, aplicar un escalador asegura convergencia rápida durante el entrenamiento. Se prefiere `RobustScaler` porque utiliza la mediana y el rango intercuartílico, volviéndolo resistente a posibles valores atípicos (*outliers*) presentes en las encuestas.

In [5]:
# RobustScaler eliminado de esta etapa por la misma razón.
# El escalado en el Pipeline usará los parámetros (mediana, IQR) de X_train únicamente.
print("Escalado delegado al Pipeline de sklearn. Sin operaciones fit aquí.")

Escalado delegado al Pipeline de sklearn. Sin operaciones fit aquí.


## 5. Exportación de Matrices de Entrenamiento

**¿Qué hace este bloque?**
Guarda las matrices finales $X$ (escalada y codificada), $y\_reg$ y $y\_clf$ en la carpeta de procesados.

**¿Por qué lo hacemos así?**
Estas matrices ya están completamente listas para ser introducidas en los algoritmos de Machine Learning. Aislar este paso permite experimentar con distintos hiperparámetros en el siguiente notebook sin tener que recalcular el One-Hot Encoding y el escalado cada vez.

In [6]:
from sklearn.model_selection import train_test_split

# El split se realiza sobre X en CRUDO (sin transformaciones) para que ningún
# estimador haya visto los datos de val/test antes del entrenamiento.
# stratify=y_clf preserva la proporción de la clase minoritaria (~11-15%) en
# cada partición, requisito crítico con desbalance 7:1.

# 1. Primer corte: 70% Entrenamiento y 30% temporal
X_train, X_temp, y_reg_train, y_reg_temp, y_clf_train, y_clf_temp = train_test_split(
    X, y_reg, y_clf,
    test_size=0.30,
    random_state=42,
    stratify=y_clf
)

# 2. Segundo corte: 20% validación y 10% test del total
X_val, X_test, y_reg_val, y_reg_test, y_clf_val, y_clf_test = train_test_split(
    X_temp, y_reg_temp, y_clf_temp,
    test_size=1/3,
    random_state=42,
    stratify=y_clf_temp
)

# 3. Guardado de matrices en crudo (el Pipeline hará fit solo sobre X_train)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

y_reg_train.to_csv('../data/processed/y_reg_train.csv', index=False)
y_reg_val.to_csv('../data/processed/y_reg_val.csv', index=False)
y_reg_test.to_csv('../data/processed/y_reg_test.csv', index=False)

y_clf_train.to_csv('../data/processed/y_clf_train.csv', index=False)
y_clf_val.to_csv('../data/processed/y_clf_val.csv', index=False)
y_clf_test.to_csv('../data/processed/y_clf_test.csv', index=False)

# También guardamos X completo sin transformar para referencia EDA
X.to_csv('../data/processed/X_features.csv', index=False)

# 4. Verificación
total = len(X_train) + len(X_val) + len(X_test)
print(f"Split estratificado completado. Total: {total} registros.")
print(f"  Train : {len(X_train):>4} ({len(X_train)/total:.0%}) — pos_clf: {y_clf_train.sum()} ({y_clf_train.mean():.1%})")
print(f"  Val   : {len(X_val):>4} ({len(X_val)/total:.0%}) — pos_clf: {y_clf_val.sum()} ({y_clf_val.mean():.1%})")
print(f"  Test  : {len(X_test):>4} ({len(X_test)/total:.0%}) — pos_clf: {y_clf_test.sum()} ({y_clf_test.mean():.1%})")
print(f"\nColumnas en X (crudo): {X_train.shape[1]}")
print("Matrices exportadas. Listas para encapsular en Pipeline en el notebook 03.")

Split estratificado completado. Total: 1915 registros.
  Train : 1340 (70%) — pos_clf: 217 (16.2%)
  Val   :  383 (20%) — pos_clf: 62 (16.2%)
  Test  :  192 (10%) — pos_clf: 31 (16.1%)

Columnas en X (crudo): 46
Matrices exportadas. Listas para encapsular en Pipeline en el notebook 03.
